# WorldKnower baseline training (ResNet18)
Runtime > Change runtime type > **GPU** before running. Edit the CONFIG cell, then run top to bottom.

In [ ]:
!nvidia-smi -L

In [ ]:
# CONFIG - edit these
DRIVE_ROOT = "/content/drive/MyDrive/WorldKnowerDrive"   # folder that contains Data/ (checkpoints/ goes next to it)
BRANCH     = "balaji/baseline-run"                        # use "main" once the dataset.py fix is merged
EPOCHS, LR, BATCH = 10, 1e-4, 32
RUN_ID     = "baseline-01"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os; print(os.listdir(DRIVE_ROOT)); print(os.listdir(f"{DRIVE_ROOT}/Data"))

In [ ]:
# Clone / update repo (safe: fetch + checkout, never reset)
import os
if not os.path.isdir("/content/WorldKnower"):
    !git clone https://github.com/TechBroMoho/WorldKnower /content/WorldKnower
%cd /content/WorldKnower
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -1

In [ ]:
# Extract images from the zip in Data/raw (skip if already done)
import glob, os
zips = glob.glob(f"{DRIVE_ROOT}/Data/raw/*.zip"); print(zips)
if not os.path.isdir("/content/data"):
    !mkdir -p /content/data && unzip -q "{zips[0]}" -d /content/data
for root, dirs, files in os.walk("/content/data"):
    print(root, dirs[:5]); break

In [ ]:
# Find the folder that holds the country folders (e.g. .../compressed_dataset)
import os
IMAGE_ROOT = None
for root, dirs, files in os.walk("/content/data"):
    if "Japan" in dirs and "France" in dirs:
        IMAGE_ROOT = root; break
print("IMAGE_ROOT =", IMAGE_ROOT)

In [ ]:
# Convert team split CSVs -> path,country with paths relative to IMAGE_ROOT
import pandas as pd, os
SPLIT_DIR = f"{DRIVE_ROOT}/Data/splits"; OUT = "/content/splits"; os.makedirs(OUT, exist_ok=True)
print(os.listdir(SPLIT_DIR))
PREFIX = "data/raw/compressed_dataset/"
for name in ["train", "val", "test"]:
    src = [f for f in os.listdir(SPLIT_DIR) if f.endswith(f"{name}.csv")][0]
    df = pd.read_csv(f"{SPLIT_DIR}/{src}")
    df = df.rename(columns={"image_path": "path"})
    df["path"] = df["path"].str.replace("\\", "/", regex=False).str.replace(PREFIX, "", regex=False)
    df[["path", "country"]].to_csv(f"{OUT}/{name}.csv", index=False)
    print(name, len(df), df.country.nunique())
# sanity: every image exists
missing = [p for p in pd.read_csv(f"{OUT}/train.csv").path if not os.path.isfile(f"{IMAGE_ROOT}/{p}")]
print("missing train images:", len(missing), missing[:3])

In [ ]:
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
%cd /content/WorldKnower
!python -u -m src.train --device cuda \
    --train-csv /content/splits/train.csv --val-csv /content/splits/val.csv \
    --dataset-root "{IMAGE_ROOT}" --checkpoint-dir "{CKPT_DIR}" \
    --history-path "{CKPT_DIR}/training_history.csv" \
    --epochs {EPOCHS} --lr {LR} --batch-size {BATCH} --num-workers 2

In [ ]:
# Best val accuracy -> paste into results/experiment_log.csv
import pandas as pd, datetime
h = pd.read_csv(f"{CKPT_DIR}/training_history.csv"); best = h.val_accuracy.max()
print(h)
print(f"{RUN_ID},{datetime.date.today()},Bal207,resnet18,{EPOCHS},{BATCH},{LR},{best:.4f},,pretrained; aug on; amp on; seed 42")